# 次日买入、后日卖出的收益标签

基于处理后的股票面板生成 `return_t2.parquet`。`trade_date` 表示信号日，标签是事后收益，供因子评价和因子收益回归使用。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

## 文件路径与参数

自动向上定位项目根目录，使用项目相对路径读写文件；内核工作目录应位于本项目内。下方同时定义本步骤使用的参数。


In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'environment.yml').is_file() and (path / 'src').is_dir()
)

PANEL = PROJECT_ROOT / "data/processed/panel.parquet"
RETURN_T2_OUTPUT = PROJECT_ROOT / "src/return_t2/return_t2.parquet"


## 读取股票面板

使用上游已按日期排序的面板，按股票对开盘价移动市场交易日位置。


In [ ]:
panel = pd.read_parquet(PANEL)

## 构建收益标签

信号日为 $t$，在 $t+1$ 开盘买入、$t+2$ 开盘卖出：

$$
\mathrm{return\_t2}_{i,t}=\frac{O^{hfq}_{i,t+2}}{O^{hfq}_{i,t+1}}-1.
$$

开盘价不填充，也不跳过停牌日寻找下一个有效价格。任一所需价格缺失或末端不足两日时，标签为空。


In [ ]:
return_t2 = panel[['trade_date','ts_code','open_hfq']].copy()

In [ ]:
return_t2['open_hfq_t1'] = return_t2.groupby('ts_code')['open_hfq'].shift(-1)
return_t2['open_hfq_t2'] = return_t2.groupby('ts_code')['open_hfq'].shift(-2)
return_t2['return_t2'] = return_t2['open_hfq_t2'] / return_t2['open_hfq_t1'] -1

## 整理输出

删除计算使用的价格列，只保留 `trade_date`、`ts_code`、`return_t2`。末端标签缺失是正常情况。


In [ ]:
return_t2 = return_t2.drop(columns = ['open_hfq', 'open_hfq_t1', 'open_hfq_t2'])

In [ ]:
return_t2.tail()

## 保存收益标签

写入 `RETURN_T2_OUTPUT`。标签包含未来实现的收益，不可用于回测当日选股资格筛选。


In [ ]:
return_t2.to_parquet(RETURN_T2_OUTPUT,index = False)